In [1]:
import requests
import zipfile
import io
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
# ==========================================
# 1. CONFIGURATION
# ==========================================
CONGRESS_NUM = "119"
BILL_TYPES = ["hconres", "hr", "hres", "hjres", "s", "sconres", "sres", "sjres"]
BASE_URL = "https://www.govinfo.gov/bulkdata/BILLSTATUS/{}/{}/BILLSTATUS-{}-{}.zip"

# ==========================================
# 2. MODIFIED EXTRACTION FUNCTION
# ==========================================
def parse_bill_cosponsors(xml_content):
    try:
        root = ET.fromstring(xml_content)
        bill = root.find('bill')

        if bill is None:
            return []

        def get_text(node, path):
            found = node.find(path)
            return found.text if found is not None else None

        # --- A. GENERAL BILL DATA ---
        latest_action_node = bill.find('actions/item')

        l_code = None
        l_text = None
        l_type = None

        if latest_action_node is not None:
            l_code = get_text(latest_action_node, 'actionCode')
            l_text = get_text(latest_action_node, 'text')
            l_type = get_text(latest_action_node, 'type')

        base_data = {
            'Congress': get_text(bill, 'congress'),
            'Number': get_text(bill, 'number'),
            'Type': get_text(bill, 'type'),
            'Origin_Chamber': get_text(bill, 'originChamber'),
            'Introduced_Date': get_text(bill, 'introducedDate'), # <--- CHANGE: column added
            'latest_action_code': l_code,
            'latest_action_type': l_type,
            'latest_action_text': l_text
        }

        # --- B. PRIMARY SPONSOR DATA ---
        sponsor_node = bill.find('sponsors/item')
        if sponsor_node is not None:
            base_data['sponsor_name'] = get_text(sponsor_node, 'fullName')
            base_data['sponsor_bioguide'] = get_text(sponsor_node, 'bioguideId')
            base_data['sponsor_party'] = get_text(sponsor_node, 'party')
        else:
            base_data.update({'sponsor_name': None, 'sponsor_bioguide': None, 'sponsor_party': None})

        # --- C. COSPONSOR DATA ---
        rows = []
        cosponsors_list = bill.findall('cosponsors/item')

        if len(cosponsors_list) > 0:
            for cosponsor in cosponsors_list:
                row = base_data.copy()
                row['cosponsor_name'] = get_text(cosponsor, 'fullName')
                row['cosponsor_bioguide'] = get_text(cosponsor, 'bioguideId')
                row['cosponsor_party'] = get_text(cosponsor, 'party')
                row['cosponsorship_date'] = get_text(cosponsor, 'sponsorshipDate')
                rows.append(row)
        else:
            row = base_data.copy()
            row.update({
                'cosponsor_name': None, 'cosponsor_bioguide': None,
                'cosponsor_party': None, 'cosponsorship_date': None
            })
            rows.append(row)

        return rows

    except Exception as e:
        print(f"Error parseando XML: {e}")
        return []

# ==========================================
# 3. MAIN PROCESS
# ==========================================
all_rows = []
print(f"Starting download and extraction for Congress {CONGRESS_NUM}...\n")

for b_type in BILL_TYPES:
    download_url = BASE_URL.format(CONGRESS_NUM, b_type, CONGRESS_NUM, b_type)
    print(f"--> Processing: {b_type.upper()}")

    try:
        response = requests.get(download_url, stream=True)
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                xml_files = [f for f in z.namelist() if f.endswith('.xml')]
                for filename in xml_files:
                    with z.open(filename) as f:
                        all_rows.extend(parse_bill_cosponsors(f.read()))
        else:
            print(f"    [!] No disponible (Status {response.status_code})")
    except Exception as e:
        print(f"    Error en ZIP {b_type}: {e}")

# ==========================================
# 4. FINAL DATAFRAME
# ==========================================
df = pd.DataFrame(all_rows)

cols_order = [
    'Congress', 'Number', 'Type', 'Origin_Chamber', 'Introduced_Date', # <--- CHANGE: included in the ordering
    'latest_action_code', 'latest_action_type', 'latest_action_text',
    'sponsor_name', 'sponsor_bioguide', 'sponsor_party',
    'cosponsor_name', 'cosponsor_bioguide', 'cosponsor_party', 'cosponsorship_date'
]
df = df[[c for c in cols_order if c in df.columns]]

print(f"\nTotal rows generated: {len(df)}")
print(df.head(10))

# df.to_csv("cosponsors_119_detailed.csv", index=False)

Iniciando descarga y extracción para el Congreso 119...

--> Procesando: HCONRES
--> Procesando: HR
--> Procesando: HRES
--> Procesando: HJRES
--> Procesando: S
--> Procesando: SCONRES
--> Procesando: SRES
--> Procesando: SJRES

Total de filas generadas: 157596
  Congress Number     Type Origin_Chamber Introduced_Date latest_action_code  \
0      119      2  HCONRES          House      2025-01-03             H11100   
1      119      1  HCONRES          House      2025-01-03               None   
2      119      3  HCONRES          House      2025-01-13             H11100   
3      119      3  HCONRES          House      2025-01-13             H11100   
4      119      3  HCONRES          House      2025-01-13             H11100   
5      119      3  HCONRES          House      2025-01-13             H11100   
6      119      3  HCONRES          House      2025-01-13             H11100   
7      119      3  HCONRES          House      2025-01-13             H11100   
8      119      3 

In [2]:
df = df.dropna(subset=['cosponsor_party'])


In [3]:
df

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13
5,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13
6,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157590,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Wyden, Ron [D-OR]",W000779,D,2026-03-02
157591,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Cantwell, Maria [D-WA]",C000127,D,2026-03-02
157592,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Booker, Cory A. [D-NJ]",B001288,D,2026-03-04
157593,119,181,SJRES,Senate,2026-04-13,None,IntroReferral,Read twice and referred to the Committee on Fo...,"Sen. Gillibrand, Kirsten E. [D-NY]",G000555,D,"Sen. Schumer, Charles E. [D-NY]",S000148,D,2026-04-16


In [4]:
democrats_list = ['S000033', 'K000383']
republicans_list = ['K000401']

df.loc[df['sponsor_bioguide'].isin(democrats_list), 'sponsor_party'] = 'D'
df.loc[df['sponsor_bioguide'].isin(republicans_list), 'sponsor_party'] = 'R'

df.loc[df['cosponsor_bioguide'].isin(democrats_list), 'cosponsor_party'] = 'D'
df.loc[df['cosponsor_bioguide'].isin(republicans_list), 'cosponsor_party'] = 'R'

In [5]:
df

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13
5,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13
6,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157590,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Wyden, Ron [D-OR]",W000779,D,2026-03-02
157591,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Cantwell, Maria [D-WA]",C000127,D,2026-03-02
157592,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Booker, Cory A. [D-NJ]",B001288,D,2026-03-04
157593,119,181,SJRES,Senate,2026-04-13,None,IntroReferral,Read twice and referred to the Committee on Fo...,"Sen. Gillibrand, Kirsten E. [D-NY]",G000555,D,"Sen. Schumer, Charles E. [D-NY]",S000148,D,2026-04-16


In [6]:
df['bill_code'] = df['Type'] + '-' + df['Number'].astype(str)

In [7]:
# Convert sponsorshipDate to datetime if it isn't already
df['cosponsorship_date'] = pd.to_datetime(df['cosponsorship_date'], errors='coerce')
df['Introduced_Date'] = pd.to_datetime(df['Introduced_Date'], errors='coerce')
# Get the earliest (minimum) date per bill_code
first_sponsorship = df.groupby('bill_code')['cosponsorship_date'].min().reset_index()
first_sponsorship.columns = ['bill_code', 'first_cosponsorshipDate']

# Merge with the original dataframe
df_final = df.merge(first_sponsorship, on='bill_code', how='left')

# View the result
df_final.head(100)

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate
0,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13,HCONRES-3,2025-01-13
1,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13,HCONRES-3,2025-01-13
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13,HCONRES-3,2025-01-13
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13,HCONRES-3,2025-01-13
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13,HCONRES-3,2025-01-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Vargas, Juan [D-CA-52]",V000130,D,2025-04-24,HCONRES-4,2025-01-28
96,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Thompson, Glenn [R-PA-15]",T000467,R,2025-04-24,HCONRES-4,2025-01-28
97,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Wagner, Ann [R-MO-2]",W000812,R,2025-05-05,HCONRES-4,2025-01-28
98,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Moran, Nathaniel [R-TX-1]",M001224,R,2025-05-19,HCONRES-4,2025-01-28


In [8]:
df_final['days_to_join'] = df_final['cosponsorship_date'] - df_final['Introduced_Date']
df_final['days_to_join'] = df_final['days_to_join'].dt.days
df_final.head()

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join
0,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13,HCONRES-3,2025-01-13,0
1,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13,HCONRES-3,2025-01-13,0
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13,HCONRES-3,2025-01-13,0
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13,HCONRES-3,2025-01-13,0
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13,HCONRES-3,2025-01-13,0


In [9]:
df_final.head(100)

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join
0,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13,HCONRES-3,2025-01-13,0
1,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13,HCONRES-3,2025-01-13,0
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13,HCONRES-3,2025-01-13,0
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13,HCONRES-3,2025-01-13,0
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13,HCONRES-3,2025-01-13,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Vargas, Juan [D-CA-52]",V000130,D,2025-04-24,HCONRES-4,2025-01-28,86
96,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Thompson, Glenn [R-PA-15]",T000467,R,2025-04-24,HCONRES-4,2025-01-28,86
97,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Wagner, Ann [R-MO-2]",W000812,R,2025-05-05,HCONRES-4,2025-01-28,97
98,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Moran, Nathaniel [R-TX-1]",M001224,R,2025-05-19,HCONRES-4,2025-01-28,111


In [10]:
df_final['days_to_join'].describe()

,days_to_join
count,154182.000000
mean,46.783172
std,92.361058
min,0.000000
25%,0.000000
50%,0.000000
75%,42.000000
max,511.000000


In [11]:
df_final['is_opposite_party'] = df_final['sponsor_party'] != df_final['cosponsor_party']

In [12]:
df_final.head()

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party
0,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13,HCONRES-3,2025-01-13,0,False
1,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13,HCONRES-3,2025-01-13,0,False
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13,HCONRES-3,2025-01-13,0,False
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13,HCONRES-3,2025-01-13,0,False
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13,HCONRES-3,2025-01-13,0,False


In [13]:
df_final['bill_code'].nunique()

12903

In [14]:
df_final['bill_code'][df_final['is_opposite_party']==1].unique()

array(['HCONRES-4', 'HCONRES-7', 'HCONRES-9', ..., 'SJRES-96', 'SJRES-98',
       'SJRES-104'], dtype=object)

In [15]:
# Get the unique codes where is_opposite_party is 1
codigos_oposicion = df_final[df_final['is_opposite_party'] == 1]['bill_code'].unique()

In [16]:
# Filter the dataframe to contain only those codes
df_verificacion = df_final[df_final['bill_code'].isin(codigos_oposicion)]

# Count how many unique codes belong to each party
conteo_partidos = df_verificacion[df_verificacion['Origin_Chamber']=='House'].groupby('sponsor_party')['bill_code'].nunique()

print(conteo_partidos)

sponsor_party
D    1966
R    2335
Name: bill_code, dtype: int64


In [17]:
mask_cumple = (df_final['is_opposite_party'] == True) & (df_final['days_to_join'] == 0)
bills_validos = df_final[mask_cumple]['bill_code'].unique()

In [18]:
df_final_b = df_final[df_final['bill_code'].isin(bills_validos)].copy()

In [19]:
# Bills that have the counterpart at any time
bills_con_opposite = df_final[df_final['is_opposite_party'] == True]['bill_code'].unique()

# Bills that have the counterpart specifically on day 0
bills_dia_0 = df_final[(df_final['is_opposite_party'] == True) & (df_final['days_to_join'] == 0)]['bill_code'].unique()

# The difference: those in the first but NOT in the second
bills_validos = set(bills_con_opposite) - set(bills_dia_0)

df_final_b_no_originals_bills = df_final[df_final['bill_code'].isin(bills_validos)].copy()

In [20]:
df_final_b_no_originals_bills.head()

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party
129,119,7,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Rules.,"Rep. Williams, Nikema [D-GA-5]",W000788,D,"Rep. Timmons, William R. [R-SC-4]",T000480,R,2025-01-31,HCONRES-7,2025-01-31,3,True
130,119,7,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Rules.,"Rep. Williams, Nikema [D-GA-5]",W000788,D,"Rep. Nunn, Zachary [R-IA-3]",N000193,R,2025-04-24,HCONRES-7,2025-01-31,86,True
723,119,30,HCONRES,House,2025-05-06,None,Floor,Star Print ordered on the referred concurrent ...,"Rep. Ezell, Mike [R-MS-4]",E000235,R,"Rep. Nehls, Troy E. [R-TX-22]",N000026,R,2025-05-06,HCONRES-30,2025-05-06,0,False
724,119,30,HCONRES,House,2025-05-06,None,Floor,Star Print ordered on the referred concurrent ...,"Rep. Ezell, Mike [R-MS-4]",E000235,R,"Rep. Malliotakis, Nicole [R-NY-11]",M000317,R,2025-05-07,HCONRES-30,2025-05-06,1,False
725,119,30,HCONRES,House,2025-05-06,None,Floor,Star Print ordered on the referred concurrent ...,"Rep. Ezell, Mike [R-MS-4]",E000235,R,"Rep. Tenney, Claudia [R-NY-24]",T000478,R,2025-05-07,HCONRES-30,2025-05-06,1,False


In [21]:

df_opp = df_final_b_no_originals_bills[df_final_b_no_originals_bills['is_opposite_party'] == True].copy()


df_opp['min_day_opp'] = df_opp.groupby('bill_code')['days_to_join'].transform('min')


df_opp['credit'] = np.where(
    df_opp['days_to_join'] == df_opp['min_day_opp'],
    0.75,
    0.25
)
df_opp = df_opp.drop(columns=['min_day_opp'])
df_first_opp_all = df_opp.copy()



In [22]:
df_first_opp_all.head()

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party,credit
129,119,7,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Rules.,"Rep. Williams, Nikema [D-GA-5]",W000788,D,"Rep. Timmons, William R. [R-SC-4]",T000480,R,2025-01-31,HCONRES-7,2025-01-31,3,True,0.75
130,119,7,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Rules.,"Rep. Williams, Nikema [D-GA-5]",W000788,D,"Rep. Nunn, Zachary [R-IA-3]",N000193,R,2025-04-24,HCONRES-7,2025-01-31,86,True,0.25
743,119,30,HCONRES,House,2025-05-06,None,Floor,Star Print ordered on the referred concurrent ...,"Rep. Ezell, Mike [R-MS-4]",E000235,R,"Rep. Riley, Josh [D-NY-19]",R000622,D,2025-05-08,HCONRES-30,2025-05-06,2,True,0.75
752,119,30,HCONRES,House,2025-05-06,None,Floor,Star Print ordered on the referred concurrent ...,"Rep. Ezell, Mike [R-MS-4]",E000235,R,"Rep. Craig, Angie [D-MN-2]",C001119,D,2025-05-13,HCONRES-30,2025-05-06,7,True,0.25
757,119,30,HCONRES,House,2025-05-06,None,Floor,Star Print ordered on the referred concurrent ...,"Rep. Ezell, Mike [R-MS-4]",E000235,R,"Rep. Perez, Marie Gluesenkamp [D-WA-3]",G000600,D,2025-05-13,HCONRES-30,2025-05-06,7,True,0.25


In [23]:
len(df_first_opp_all)

1170

In [24]:
df_final_b.head()


,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party
22,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Moore, Gwen [D-WI-4]",M001160,D,2025-01-28,HCONRES-4,2025-01-28,0,True
23,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Feenstra, Randy [R-IA-4]",F000446,R,2025-01-28,HCONRES-4,2025-01-28,0,False
24,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Sewell, Terri A. [D-AL-7]",S001185,D,2025-01-28,HCONRES-4,2025-01-28,0,True
25,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Carey, Mike [R-OH-15]",C001126,R,2025-01-28,HCONRES-4,2025-01-28,0,False
26,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Boyle, Brendan F. [D-PA-2]",B001296,D,2025-01-28,HCONRES-4,2025-01-28,0,True


In [25]:
df_final_b['bill_code'][df_final_b['is_opposite_party']==1].nunique()

6283

In [26]:
df_final_b['credit'] = 0.2


# Day 0: 1.2 points
df_final_b.loc[(df_final_b['is_opposite_party'] == True) & (df_final_b['days_to_join'] == 0), 'credit'] = 1

# Day 1-5: 0.5 points
df_final_b.loc[(df_final_b['is_opposite_party'] == True) & (df_final_b['days_to_join'] > 0) & (df_final_b['days_to_join'] <= 5), 'credit'] = 0.5

df_final_b.loc[(df_final_b['is_opposite_party'] == True) & (df_final_b['days_to_join'] < 5), 'credit'] = 0.25

# Day 0: 1.0 points
df_final_b.loc[(df_final_b['is_opposite_party'] == False) & (df_final_b['days_to_join'] == 0), 'credit'] = 0.5

# Day 1-5: 0.5 points
df_final_b.loc[(df_final_b['is_opposite_party'] == False) & (df_final_b['days_to_join'] > 0) & (df_final_b['days_to_join'] <= 5), 'credit'] = 0.25

df_final_b.loc[(df_final_b['is_opposite_party'] == False) & (df_final_b['days_to_join'] < 5), 'credit'] = 0.125

In [27]:
weight_map = {
    'HR': 1.0, 'S': 1.0,
    'HJRES': 1.0, 'SJRES': 1.0,
    'HCONRES': 0.3, 'SCONRES': 0.3,
    'HRES': 0.1, 'SRES': 0.1
}

In [28]:
df_final_b['weight_type'] = df_final_b['Type'].map(weight_map).fillna(0)

In [29]:
df_first_opp_all['weight_type'] = df_first_opp_all['Type'].map(weight_map).fillna(0)

In [30]:
df_final_b['weight_type'].value_counts()

,count
weight_type,
1.0,60415
0.1,12239
0.3,1014


In [31]:
df_first_opp_all['total'] = df_first_opp_all['weight_type'] * df_first_opp_all['credit']

In [32]:
df_final_b['total'] = df_final_b['weight_type'] * df_final_b['credit']

In [33]:
len(df_first_opp_all)

1170

In [34]:
len(df_final_b)

73668

In [35]:
df_final_b = pd.concat([df_final_b, df_first_opp_all], ignore_index=True)

In [36]:
len(df_final_b)

74838

In [37]:
df_final_b.head()

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,...,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party,credit,weight_type,total
0,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,M001160,D,2025-01-28,HCONRES-4,2025-01-28,0,True,0.250,0.3,0.0750
1,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,F000446,R,2025-01-28,HCONRES-4,2025-01-28,0,False,0.125,0.3,0.0375
2,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,S001185,D,2025-01-28,HCONRES-4,2025-01-28,0,True,0.250,0.3,0.0750
3,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,C001126,R,2025-01-28,HCONRES-4,2025-01-28,0,False,0.125,0.3,0.0375
4,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,B001296,D,2025-01-28,HCONRES-4,2025-01-28,0,True,0.250,0.3,0.0750


In [38]:
df_final_b['bill_code'].nunique()

6751

In [39]:
df_final_b['total'][df_final_b['cosponsor_party']=='D'].value_counts()

,count
total,
0.2000,21584
0.1250,7456
0.2500,7423
0.0125,3729
0.0250,2408
0.0200,1137
0.7500,349
0.0375,322
0.0600,188


In [40]:
df_final_b['total'][df_final_b['cosponsor_party']=='R'].value_counts()

,count
total,
0.2000,13503
0.1250,6428
0.2500,4510
0.0125,3082
0.0250,1391
0.0200,511
0.0375,190
0.7500,137
0.0600,136


In [41]:
df_final_b[(df_final_b['cosponsor_bioguide']=='O000174')]

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,...,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party,credit,weight_type,total
59050,119,555,S,Senate,2025-02-12,None,Calendars,Placed on Senate Legislative Calendar under Ge...,"Sen. Kaine, Tim [D-VA]",K000384,...,O000174,D,2025-09-04,S-555,2025-02-12,204,False,0.200,1.0,0.200
59077,119,138,S,Senate,2025-01-16,None,IntroReferral,Read twice and referred to the Committee on Ba...,"Sen. Sheehy, Tim [R-MT]",S001232,...,O000174,D,2025-09-10,S-138,2025-01-16,237,True,0.200,1.0,0.200
59173,119,342,S,Senate,2025-01-30,None,Committee,Committee on Veterans' Affairs. Ordered to be ...,"Sen. Murray, Patty [D-WA]",M001111,...,O000174,D,2025-09-11,S-342,2025-01-30,224,False,0.200,1.0,0.200
59283,119,280,S,Senate,2025-01-28,None,IntroReferral,Read twice and referred to the Committee on Fo...,"Sen. Shaheen, Jeanne [D-NH]",S001181,...,O000174,D,2025-01-28,S-280,2025-01-28,0,False,0.125,1.0,0.125
59332,119,2947,S,Senate,2025-09-30,None,IntroReferral,Read twice and referred to the Committee on Ho...,"Sen. Hassan, Margaret Wood [D-NH]",H001076,...,O000174,D,2025-12-18,S-2947,2025-09-30,79,False,0.200,1.0,0.200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74715,119,373,S,Senate,2025-02-03,None,IntroReferral,Read twice and referred to the Committee on Ho...,"Sen. Lankford, James [R-OK]",L000575,...,O000174,D,2025-05-05,S-373,2025-02-03,91,True,0.250,1.0,0.250
74731,119,1498,S,Senate,2025-04-28,None,Calendars,Placed on Senate Legislative Calendar under Ge...,"Sen. Hawley, Josh [R-MO]",H001089,...,O000174,D,2025-07-30,S-1498,2025-04-28,93,True,0.750,1.0,0.750
74789,119,1919,S,Senate,2025-05-22,None,IntroReferral,Read twice and referred to the Committee on Fi...,"Sen. Hyde-Smith, Cindy [R-MS]",H001079,...,O000174,D,2025-12-11,S-1919,2025-05-22,203,True,0.750,1.0,0.750
74799,119,698,S,Senate,2025-02-24,None,IntroReferral,Read twice and referred to the Committee on th...,"Sen. McConnell, Mitch [R-KY]",M000355,...,O000174,D,2025-04-02,S-698,2025-02-24,37,True,0.750,1.0,0.750


In [42]:
# Group and compute statistics
cosponsor_summary = (
    df_final_b.groupby('cosponsor_bioguide')
    .agg({
        'total': 'sum',
        'bill_code': 'count'
    })
    .reset_index()
)

# Rename
cosponsor_summary.columns = ['bioguide_id', 'num_cross_party_cosponsored_bills', 'num_rows']

# Sort
cosponsor_summary = cosponsor_summary.sort_values('num_cross_party_cosponsored_bills', ascending=False).reset_index(drop=True)

cosponsor_summary.head(20)

,bioguide_id,num_cross_party_cosponsored_bills,num_rows
0,F000466,211.6100,1052
1,V000138,159.6750,756
2,L000599,143.8025,739
3,G000583,116.6750,591
4,D000230,111.2175,552
5,B001298,89.4450,503
6,N000147,84.3525,538
7,S001201,83.3250,439
8,N000191,74.3350,373
9,P000613,71.9950,371


In [43]:
cosponsor_summary[cosponsor_summary['bioguide_id']=='O000174']

,bioguide_id,num_cross_party_cosponsored_bills,num_rows
128,O000174,29.0025,151


In [44]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [45]:
# Export
cosponsor_summary.to_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temp_Files/Temp_B/Temp_B_v1.7.0_2026_06_02.csv',
    index=False
)